# 03 — Generate KoVAE Synthetic Subjects (Dual Methods)

This notebook keeps **both** KoVAE generation approaches:

```text
1. rollout_v1
   - sample z0
   - stabilized Koopman rollout
   - decode

2. posterior_bank_v2
   - sample full posterior latent trajectory from train windows
   - interpolate / perturb trajectory
   - optional Koopman-guided blending
   - decode
```

This lets you keep both implementations and compare them later using realism/diversity metrics and the downstream activity classifier.

## Main idea

```text
data/synthetic_subjects/kovae/rollout_v1/
data/synthetic_subjects/kovae/posterior_bank_v2/

results/kovae_generation/rollout_v1/
results/kovae_generation/posterior_bank_v2/

figures/kovae_generation/rollout_v1/
figures/kovae_generation/posterior_bank_v2/
```

## Plot switch

Use:

```python
GEN_CONFIG.save_plot = True
```

to save plots, or:

```python
GEN_CONFIG.save_plot = False
```

to only show plots.


In [1]:

# ============================================================
# 03_generate_kovae_synthetic_subjects_dual_methods.py
#
# Keep both KoVAE generation approaches:
#   1) rollout_v1
#   2) posterior_bank_v2
#
# Each method is saved into its own subfolders.
# ============================================================

from __future__ import annotations

import json
import logging
import random
from dataclasses import asdict, dataclass, field, fields
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset


# ============================================================
# Training config structure from Notebook 02
# ============================================================

@dataclass
class KoVAETrainConfig:
    project_root: str = "/home/iailab42/khans1/projects/ir"
    processed_subdir: str = "data/processed/native_rates"

    random_seed: int = 42

    train_subjects: List[str] = field(default_factory=lambda: [
        "S1", "S2", "S3", "S4", "S5", "S6", "S9", "S11", "S12", "S13"
    ])
    val_subjects: List[str] = field(default_factory=lambda: ["S14", "S15"])
    test_subjects: List[str] = field(default_factory=lambda: ["S7", "S8", "S10"])

    acc_hz: int = 32
    bvp_hz: int = 64
    slow_hz: int = 4

    acc_len: int = 256
    bvp_len: int = 512
    slow_len: int = 32

    acc_channels: int = 3
    bvp_channels: int = 1
    slow_channels: int = 2

    latent_steps: int = 64
    branch_hidden_dim: int = 64
    fusion_hidden_dim: int = 128
    latent_dim: int = 32

    activity_embedding_dim: int = 16
    use_subject_condition: bool = True
    subject_embedding_dim: int = 16
    subject_dropout_prob: float = 0.20

    dropout: float = 0.10

    batch_size: int = 64
    num_workers: int = 0
    epochs: int = 100
    learning_rate: float = 1e-3
    weight_decay: float = 1e-5
    patience: int = 15
    gradient_clip_norm: float = 1.0

    use_weighted_sampler: bool = True
    use_amp: bool = True

    reconstruction_weight_bvp: float = 1.0
    reconstruction_weight_acc: float = 1.0
    reconstruction_weight_slow: float = 1.0

    alpha_koopman: float = 0.10
    beta_kl: float = 1e-3
    koopman_ridge: float = 1e-3

    max_reconstruction_examples: int = 3


# ============================================================
# Generation config
# ============================================================

@dataclass
class GenerationConfig:
    project_root: str = "/home/iailab42/khans1/projects/ir"

    checkpoint_path: str = "models/checkpoints/kovae_best.pt"
    koopman_matrix_path: str = "results/kovae/best_koopman_matrix.npy"
    processed_subdir: str = "data/processed/native_rates"

    random_seed: int = 123

    methods_to_run: List[str] = field(default_factory=lambda: [
        "rollout_v1",
        "posterior_bank_v2",
    ])

    num_synthetic_subjects: int = 10
    windows_per_subject: int = 500
    generation_batch_size: int = 128
    synthetic_subject_prefix: str = "KOVAE_SYN"

    activity_sampling_strategy: str = "train_distribution"

    stabilize_koopman: bool = True
    target_spectral_radius: float = 0.98

    # rollout_v1 parameters
    rollout_latent_noise_scale: float = 0.03
    rollout_z0_noise_scale: float = 0.05

    # posterior_bank_v2 parameters
    posterior_noise_scale: float = 0.06
    posterior_interpolation_prob: float = 0.70
    posterior_koopman_blend_weight: float = 0.15

    # subject style
    subject_style_strategy: str = "interpolate_train_subject_embeddings"
    subject_style_noise_scale: float = 0.05

    # user-requested switch
    save_plot: bool = True
    max_plot_examples: int = 4

    synthetic_base_subdir: str = "data/synthetic_subjects/kovae"
    results_base_subdir: str = "results/kovae_generation"
    figures_base_subdir: str = "figures/kovae_generation"


GEN_CONFIG = GenerationConfig()


# ============================================================
# Paths, logging, reproducibility
# ============================================================

def get_base_paths(config: GenerationConfig) -> Dict[str, Path]:
    root = Path(config.project_root)
    return {
        "project_root": root,
        "processed": root / config.processed_subdir,
        "checkpoint": root / config.checkpoint_path,
        "koopman_matrix": root / config.koopman_matrix_path,
        "configs": root / "configs",
        "logs": root / "logs",
        "synthetic_base": root / config.synthetic_base_subdir,
        "results_base": root / config.results_base_subdir,
        "figures_base": root / config.figures_base_subdir,
    }


def get_method_paths(base_paths: Dict[str, Path], method_name: str) -> Dict[str, Path]:
    return {
        "synthetic": base_paths["synthetic_base"] / method_name,
        "results": base_paths["results_base"] / method_name,
        "figures": base_paths["figures_base"] / method_name,
    }


def create_dirs(paths: Dict[str, Path]) -> None:
    for path in paths.values():
        if path.suffix == "":
            path.mkdir(parents=True, exist_ok=True)


def setup_logging(log_path: Path) -> logging.Logger:
    logger = logging.getLogger("kovae_generation_dual_methods")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()

    formatter = logging.Formatter(
        fmt="%(asctime)s | %(levelname)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    file_handler = logging.FileHandler(log_path, mode="w")
    file_handler.setFormatter(formatter)
    file_handler.setLevel(logging.INFO)

    stream_handler = logging.StreamHandler()
    stream_handler.setFormatter(formatter)
    stream_handler.setLevel(logging.INFO)

    logger.addHandler(file_handler)
    logger.addHandler(stream_handler)

    return logger


def set_random_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def save_json(data: dict, path: Path) -> None:
    path.write_text(json.dumps(data, indent=2), encoding="utf-8")


def load_checkpoint(path: Path, device: torch.device) -> dict:
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


def config_from_checkpoint(checkpoint_config: dict) -> KoVAETrainConfig:
    allowed = {field.name for field in fields(KoVAETrainConfig)}
    filtered = {key: value for key, value in checkpoint_config.items() if key in allowed}
    return KoVAETrainConfig(**filtered)


# ============================================================
# Model definitions
# Same as Notebook 02 + encode/decode helpers
# ============================================================

class BranchEncoder(nn.Module):
    def __init__(self, in_channels: int, hidden_dim: int, latent_steps: int, dropout: float) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, hidden_dim, kernel_size=7, padding=3),
            nn.GELU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=5, padding=2),
            nn.GELU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Dropout(dropout),
        )
        self.pool = nn.AdaptiveAvgPool1d(latent_steps)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.transpose(1, 2)
        h = self.net(x)
        h = self.pool(h)
        return h.transpose(1, 2)


class BranchDecoder(nn.Module):
    def __init__(
        self,
        latent_dim: int,
        condition_dim: int,
        hidden_dim: int,
        output_len: int,
        output_channels: int,
        dropout: float,
    ) -> None:
        super().__init__()
        self.output_len = output_len
        self.gru = nn.GRU(
            input_size=latent_dim + condition_dim,
            hidden_size=hidden_dim,
            batch_first=True,
        )
        self.conv = nn.Sequential(
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=5, padding=2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(hidden_dim, output_channels, kernel_size=3, padding=1),
        )

    def forward(self, z: torch.Tensor, condition_seq: torch.Tensor) -> torch.Tensor:
        h = torch.cat([z, condition_seq], dim=-1)
        h, _ = self.gru(h)
        h = h.transpose(1, 2)
        h = F.interpolate(h, size=self.output_len, mode="linear", align_corners=False)
        out = self.conv(h)
        return out.transpose(1, 2)


class MultiBranchKoVAE(nn.Module):
    def __init__(self, config: KoVAETrainConfig, num_activities: int, num_subject_tokens: int) -> None:
        super().__init__()
        self.config = config
        self.num_activities = num_activities
        self.num_subject_tokens = num_subject_tokens

        self.bvp_encoder = BranchEncoder(
            in_channels=config.bvp_channels,
            hidden_dim=config.branch_hidden_dim,
            latent_steps=config.latent_steps,
            dropout=config.dropout,
        )
        self.acc_encoder = BranchEncoder(
            in_channels=config.acc_channels,
            hidden_dim=config.branch_hidden_dim,
            latent_steps=config.latent_steps,
            dropout=config.dropout,
        )
        self.slow_encoder = BranchEncoder(
            in_channels=config.slow_channels,
            hidden_dim=config.branch_hidden_dim,
            latent_steps=config.latent_steps,
            dropout=config.dropout,
        )

        self.activity_embedding = nn.Embedding(num_activities, config.activity_embedding_dim)

        if config.use_subject_condition:
            self.subject_embedding = nn.Embedding(num_subject_tokens, config.subject_embedding_dim)
            condition_dim = config.activity_embedding_dim + config.subject_embedding_dim
        else:
            self.subject_embedding = None
            condition_dim = config.activity_embedding_dim

        self.condition_dim = condition_dim

        fusion_input_dim = 3 * config.branch_hidden_dim + condition_dim

        self.fusion_gru = nn.GRU(
            input_size=fusion_input_dim,
            hidden_size=config.fusion_hidden_dim,
            batch_first=True,
        )

        self.to_mu = nn.Linear(config.fusion_hidden_dim, config.latent_dim)
        self.to_logvar = nn.Linear(config.fusion_hidden_dim, config.latent_dim)

        self.bvp_decoder = BranchDecoder(
            latent_dim=config.latent_dim,
            condition_dim=condition_dim,
            hidden_dim=config.branch_hidden_dim,
            output_len=config.bvp_len,
            output_channels=config.bvp_channels,
            dropout=config.dropout,
        )
        self.acc_decoder = BranchDecoder(
            latent_dim=config.latent_dim,
            condition_dim=condition_dim,
            hidden_dim=config.branch_hidden_dim,
            output_len=config.acc_len,
            output_channels=config.acc_channels,
            dropout=config.dropout,
        )
        self.slow_decoder = BranchDecoder(
            latent_dim=config.latent_dim,
            condition_dim=condition_dim,
            hidden_dim=config.branch_hidden_dim,
            output_len=config.slow_len,
            output_channels=config.slow_channels,
            dropout=config.dropout,
        )

    def make_condition(self, activity: torch.Tensor, subject: torch.Tensor) -> torch.Tensor:
        activity_emb = self.activity_embedding(activity)
        if self.config.use_subject_condition:
            subject_emb = self.subject_embedding(subject)
            condition = torch.cat([activity_emb, subject_emb], dim=-1)
        else:
            condition = activity_emb
        return condition

    def encode(
        self,
        bvp: torch.Tensor,
        acc: torch.Tensor,
        slow: torch.Tensor,
        activity: torch.Tensor,
        subject: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        bvp_h = self.bvp_encoder(bvp)
        acc_h = self.acc_encoder(acc)
        slow_h = self.slow_encoder(slow)

        condition = self.make_condition(activity, subject)
        condition_seq = condition[:, None, :].repeat(1, self.config.latent_steps, 1)

        fused = torch.cat([bvp_h, acc_h, slow_h, condition_seq], dim=-1)
        fused_h, _ = self.fusion_gru(fused)

        mu = self.to_mu(fused_h)
        logvar = self.to_logvar(fused_h).clamp(min=-8.0, max=8.0)

        return mu, logvar

    def decode_with_condition_vector(self, z: torch.Tensor, condition: torch.Tensor) -> Dict[str, torch.Tensor]:
        condition_seq = condition[:, None, :].repeat(1, z.shape[1], 1)

        return {
            "bvp": self.bvp_decoder(z, condition_seq),
            "acc": self.acc_decoder(z, condition_seq),
            "slow": self.slow_decoder(z, condition_seq),
        }


# ============================================================
# Data loading
# ============================================================

def load_native_rate_arrays(processed_dir: Path) -> Dict[str, np.ndarray]:
    required_files = {
        "X_acc": "all_X_acc_32hz.npy",
        "X_bvp": "all_X_bvp_64hz.npy",
        "X_slow": "all_X_slow_4hz.npy",
        "y": "all_y.npy",
        "subjects": "all_subject.npy",
    }
    arrays = {}
    missing = []

    for key, filename in required_files.items():
        path = processed_dir / filename
        if not path.exists():
            missing.append(str(path))
        else:
            if key == "subjects":
                arrays[key] = np.load(path, allow_pickle=True).astype(str)
            else:
                arrays[key] = np.load(path)

    if missing:
        raise FileNotFoundError("Missing required preprocessing files:\n" + "\n".join(missing))

    return arrays


def build_indices_by_subject(subjects: np.ndarray, selected_subjects: List[str]) -> np.ndarray:
    mask = np.isin(subjects.astype(str), np.asarray(selected_subjects, dtype=str))
    return np.where(mask)[0].astype(np.int64)


def encode_activities(y: np.ndarray, activity_to_idx: Dict[str, int]) -> np.ndarray:
    return np.asarray([activity_to_idx[str(int(label))] for label in y], dtype=np.int64)


def encode_subjects(subjects: np.ndarray, subject_to_idx: Dict[str, int]) -> np.ndarray:
    unk = subject_to_idx.get("UNK", 0)
    return np.asarray([subject_to_idx.get(str(subject), unk) for subject in subjects], dtype=np.int64)


class NativeRatePPGDataset(Dataset):
    def __init__(self, arrays: Dict[str, np.ndarray], y_encoded: np.ndarray, subject_encoded: np.ndarray, indices: np.ndarray) -> None:
        self.X_acc = arrays["X_acc"]
        self.X_bvp = arrays["X_bvp"]
        self.X_slow = arrays["X_slow"]
        self.y_encoded = y_encoded
        self.subject_encoded = subject_encoded
        self.indices = indices.astype(np.int64)

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, item: int) -> Dict[str, torch.Tensor]:
        idx = self.indices[item]
        return {
            "acc": torch.from_numpy(self.X_acc[idx]).float(),
            "bvp": torch.from_numpy(self.X_bvp[idx]).float(),
            "slow": torch.from_numpy(self.X_slow[idx]).float(),
            "activity": torch.tensor(self.y_encoded[idx], dtype=torch.long),
            "subject": torch.tensor(self.subject_encoded[idx], dtype=torch.long),
            "global_index": torch.tensor(idx, dtype=torch.long),
        }


def move_batch_to_device(batch: Dict[str, torch.Tensor], device: torch.device) -> Dict[str, torch.Tensor]:
    return {key: value.to(device, non_blocking=True) for key, value in batch.items()}


# ============================================================
# Koopman stabilization
# ============================================================

def compute_spectral_radius(A: np.ndarray) -> float:
    eigvals = np.linalg.eigvals(A)
    return float(np.max(np.abs(eigvals)))


def stabilize_koopman_matrix(A: np.ndarray, target_radius: float, stabilize: bool) -> Tuple[np.ndarray, Dict[str, object]]:
    raw_radius = compute_spectral_radius(A)

    if stabilize and raw_radius > target_radius:
        scale = target_radius / raw_radius
        A_stable = A * scale
    else:
        scale = 1.0
        A_stable = A.copy()

    stable_radius = compute_spectral_radius(A_stable)

    summary = {
        "raw_spectral_radius": float(raw_radius),
        "target_spectral_radius": float(target_radius),
        "stable_spectral_radius": float(stable_radius),
        "stabilize_koopman": bool(stabilize),
        "scaling_factor": float(scale),
        "stability_action": "scaled_matrix" if scale != 1.0 else "unchanged",
    }

    return A_stable.astype(np.float32), summary


# ============================================================
# Latent banks
# ============================================================

def collect_activity_latent_banks(
    model: MultiBranchKoVAE,
    arrays: Dict[str, np.ndarray],
    train_config: KoVAETrainConfig,
    activity_to_idx: Dict[str, int],
    subject_to_idx: Dict[str, int],
    device: torch.device,
    logger: logging.Logger,
) -> Dict[str, Dict[int, np.ndarray]]:
    y_encoded = encode_activities(arrays["y"], activity_to_idx)
    subject_encoded = encode_subjects(arrays["subjects"], subject_to_idx)
    train_indices = build_indices_by_subject(arrays["subjects"], train_config.train_subjects)

    dataset = NativeRatePPGDataset(
        arrays=arrays,
        y_encoded=y_encoded,
        subject_encoded=subject_encoded,
        indices=train_indices,
    )

    loader = DataLoader(
        dataset,
        batch_size=train_config.batch_size,
        shuffle=False,
        num_workers=train_config.num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    z0_lists = {idx: [] for idx in sorted(set(activity_to_idx.values()))}
    trajectory_lists = {idx: [] for idx in sorted(set(activity_to_idx.values()))}

    model.eval()

    with torch.no_grad():
        for batch in loader:
            batch = move_batch_to_device(batch, device)

            mu, _ = model.encode(
                bvp=batch["bvp"],
                acc=batch["acc"],
                slow=batch["slow"],
                activity=batch["activity"],
                subject=batch["subject"],
            )

            trajectories = mu.detach().cpu().numpy().astype(np.float32)
            z0 = trajectories[:, 0, :]
            activities = batch["activity"].detach().cpu().numpy()

            for activity_idx in np.unique(activities):
                mask = activities == activity_idx
                z0_lists[int(activity_idx)].append(z0[mask])
                trajectory_lists[int(activity_idx)].append(trajectories[mask])

    z0_banks = {}
    trajectory_banks = {}

    for activity_idx, chunks in z0_lists.items():
        if len(chunks) == 0:
            continue
        z0_banks[int(activity_idx)] = np.concatenate(chunks, axis=0).astype(np.float32)

    for activity_idx, chunks in trajectory_lists.items():
        if len(chunks) == 0:
            continue
        trajectory_banks[int(activity_idx)] = np.concatenate(chunks, axis=0).astype(np.float32)

    for activity_idx in sorted(trajectory_banks.keys()):
        logger.info(
            "Activity %s | z0 bank %s | trajectory bank %s",
            activity_idx,
            z0_banks[activity_idx].shape,
            trajectory_banks[activity_idx].shape,
        )

    return {
        "z0_banks": z0_banks,
        "trajectory_banks": trajectory_banks,
    }


# ============================================================
# Sampling helpers
# ============================================================

def sample_activity_sequence(rng: np.random.Generator, y_train_encoded: np.ndarray, n: int, strategy: str) -> np.ndarray:
    unique, counts = np.unique(y_train_encoded, return_counts=True)

    if strategy == "uniform":
        probabilities = np.ones(len(unique), dtype=np.float64) / len(unique)
    elif strategy == "train_distribution":
        probabilities = counts.astype(np.float64) / counts.sum()
    else:
        raise ValueError(f"Unknown activity_sampling_strategy: {strategy}")

    return rng.choice(unique, size=n, replace=True, p=probabilities).astype(np.int64)


def sample_synthetic_subject_style(
    model: MultiBranchKoVAE,
    subject_to_idx: Dict[str, int],
    rng: np.random.Generator,
    generation_config: GenerationConfig,
    device: torch.device,
) -> Tuple[torch.Tensor, Dict[str, object]]:
    if not model.config.use_subject_condition:
        return torch.empty(0, device=device), {
            "style_strategy": "no_subject_condition",
            "style_subject_a": None,
            "style_subject_b": None,
            "style_mix_lambda": None,
        }

    subject_items = [(subject, idx) for subject, idx in subject_to_idx.items() if subject != "UNK"]
    if len(subject_items) == 0:
        raise RuntimeError("No train subject embeddings found.")

    subject_a, idx_a = subject_items[int(rng.integers(0, len(subject_items)))]
    subject_b, idx_b = subject_items[int(rng.integers(0, len(subject_items)))]
    lam = float(rng.uniform(0.25, 0.75))

    emb_weight = model.subject_embedding.weight.detach()
    emb_a = emb_weight[idx_a]
    emb_b = emb_weight[idx_b]
    style = lam * emb_a + (1.0 - lam) * emb_b

    if generation_config.subject_style_noise_scale > 0:
        train_indices = torch.tensor([idx for _, idx in subject_items], device=device, dtype=torch.long)
        emb_std = emb_weight[train_indices].std(dim=0)
        noise = torch.randn_like(style) * emb_std * generation_config.subject_style_noise_scale
        style = style + noise

    metadata = {
        "style_strategy": generation_config.subject_style_strategy,
        "style_subject_a": subject_a,
        "style_subject_b": subject_b,
        "style_mix_lambda": lam,
    }

    return style.detach(), metadata


def build_condition_from_style(model: MultiBranchKoVAE, activity: torch.Tensor, subject_style: torch.Tensor) -> torch.Tensor:
    activity_emb = model.activity_embedding(activity)
    if model.config.use_subject_condition:
        style = subject_style[None, :].repeat(activity.shape[0], 1)
        return torch.cat([activity_emb, style], dim=-1)
    return activity_emb


# ============================================================
# Generation methods
# ============================================================

def generate_latent_rollout_batch(
    activity_batch: np.ndarray,
    z0_banks: Dict[int, np.ndarray],
    A_stable: np.ndarray,
    train_config: KoVAETrainConfig,
    generation_config: GenerationConfig,
    rng: np.random.Generator,
    device: torch.device,
) -> torch.Tensor:
    batch_size = len(activity_batch)
    latent_dim = train_config.latent_dim
    latent_steps = train_config.latent_steps

    z0 = np.zeros((batch_size, latent_dim), dtype=np.float32)
    per_sample_std = np.zeros((batch_size, latent_dim), dtype=np.float32)

    available_keys = sorted(z0_banks.keys())

    for i, activity_idx in enumerate(activity_batch):
        bank = z0_banks.get(int(activity_idx))
        if bank is None or len(bank) == 0:
            bank = z0_banks[int(rng.choice(available_keys))]

        selected = bank[int(rng.integers(0, len(bank)))]
        bank_std = bank.std(axis=0).astype(np.float32)

        z0[i] = selected + generation_config.rollout_z0_noise_scale * bank_std * rng.normal(0.0, 1.0, size=latent_dim).astype(np.float32)
        per_sample_std[i] = bank_std

    A_torch = torch.from_numpy(A_stable).float().to(device)
    z = torch.zeros((batch_size, latent_steps, latent_dim), dtype=torch.float32, device=device)
    z[:, 0, :] = torch.from_numpy(z0).float().to(device)

    noise_std = torch.from_numpy(per_sample_std).float().to(device)

    for t in range(1, latent_steps):
        z[:, t, :] = z[:, t - 1, :] @ A_torch
        if generation_config.rollout_latent_noise_scale > 0:
            z[:, t, :] = z[:, t, :] + torch.randn_like(z[:, t, :]) * noise_std * generation_config.rollout_latent_noise_scale

    return z


def generate_posterior_bank_batch(
    activity_batch: np.ndarray,
    trajectory_banks: Dict[int, np.ndarray],
    A_stable: np.ndarray,
    generation_config: GenerationConfig,
    rng: np.random.Generator,
    device: torch.device,
) -> torch.Tensor:
    batch_size = len(activity_batch)
    some_bank = next(iter(trajectory_banks.values()))
    latent_steps = some_bank.shape[1]
    latent_dim = some_bank.shape[2]

    sampled = np.zeros((batch_size, latent_steps, latent_dim), dtype=np.float32)
    available_keys = sorted(trajectory_banks.keys())

    for i, activity_idx in enumerate(activity_batch):
        bank = trajectory_banks.get(int(activity_idx))
        if bank is None or len(bank) == 0:
            bank = trajectory_banks[int(rng.choice(available_keys))]

        idx1 = int(rng.integers(0, len(bank)))
        traj1 = bank[idx1].copy()

        if len(bank) >= 2 and rng.random() < generation_config.posterior_interpolation_prob:
            idx2 = int(rng.integers(0, len(bank)))
            lam = float(rng.uniform(0.25, 0.75))
            traj2 = bank[idx2]
            traj = lam * traj1 + (1.0 - lam) * traj2
        else:
            traj = traj1

        bank_std = bank.std(axis=0).astype(np.float32)
        noise = rng.normal(0.0, 1.0, size=traj.shape).astype(np.float32)
        traj = traj + generation_config.posterior_noise_scale * bank_std * noise

        sampled[i] = traj.astype(np.float32)

    z = torch.from_numpy(sampled).float().to(device)

    if generation_config.posterior_koopman_blend_weight > 0:
        A_torch = torch.from_numpy(A_stable).float().to(device)
        blend = float(generation_config.posterior_koopman_blend_weight)

        for t in range(1, z.shape[1]):
            predicted = z[:, t - 1, :] @ A_torch
            z[:, t, :] = (1.0 - blend) * z[:, t, :] + blend * predicted

    return z


def decode_synthetic_batch(
    model: MultiBranchKoVAE,
    z: torch.Tensor,
    activity_batch: np.ndarray,
    subject_style: torch.Tensor,
    device: torch.device,
) -> Dict[str, np.ndarray]:
    activity_t = torch.from_numpy(activity_batch).long().to(device)

    condition = build_condition_from_style(
        model=model,
        activity=activity_t,
        subject_style=subject_style.to(device),
    )

    with torch.no_grad():
        decoded = model.decode_with_condition_vector(z, condition)

    return {
        "bvp": decoded["bvp"].detach().cpu().numpy().astype(np.float32),
        "acc": decoded["acc"].detach().cpu().numpy().astype(np.float32),
        "slow": decoded["slow"].detach().cpu().numpy().astype(np.float32),
    }


# ============================================================
# Plot helpers
# ============================================================

def finalize_plot(fig: plt.Figure, path: Optional[Path], save_plot: bool) -> None:
    if save_plot:
        if path is None:
            raise ValueError("save_plot=True requires a path.")
        fig.savefig(path, dpi=200, bbox_inches="tight")
        plt.close(fig)
    else:
        plt.show()


def plot_koopman_eigenvalues_comparison(A_raw: np.ndarray, A_stable: np.ndarray, path: Path, save_plot: bool) -> None:
    raw_eigs = np.linalg.eigvals(A_raw)
    stable_eigs = np.linalg.eigvals(A_stable)

    theta = np.linspace(0, 2 * np.pi, 400)
    unit_x = np.cos(theta)
    unit_y = np.sin(theta)

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.plot(unit_x, unit_y, linestyle="--", label="unit circle")
    ax.scatter(raw_eigs.real, raw_eigs.imag, label="raw A", alpha=0.75)
    ax.scatter(stable_eigs.real, stable_eigs.imag, label="stable A", alpha=0.75)
    ax.axhline(0, linewidth=0.8)
    ax.axvline(0, linewidth=0.8)
    ax.set_xlabel("Real")
    ax.set_ylabel("Imaginary")
    ax.set_title("Raw vs stabilized Koopman eigenvalues")
    ax.legend()
    ax.axis("equal")
    fig.tight_layout()
    finalize_plot(fig, path, save_plot)


def plot_activity_distribution(y: np.ndarray, path: Path, save_plot: bool) -> None:
    labels, counts = np.unique(y, return_counts=True)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar([str(int(label)) for label in labels], counts)
    ax.set_xlabel("Activity label")
    ax.set_ylabel("Number of generated windows")
    ax.set_title("Synthetic activity distribution")
    fig.tight_layout()
    finalize_plot(fig, path, save_plot)


def plot_branch_histograms(real_arrays: Dict[str, np.ndarray], synthetic_arrays: Dict[str, np.ndarray], path: Path, save_plot: bool) -> None:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].hist(real_arrays["X_bvp"].reshape(-1), bins=80, alpha=0.5, density=True, label="real")
    axes[0].hist(synthetic_arrays["X_bvp"].reshape(-1), bins=80, alpha=0.5, density=True, label="synthetic")
    axes[0].set_title("BVP distribution")
    axes[0].legend()

    axes[1].hist(real_arrays["X_acc"].reshape(-1), bins=80, alpha=0.5, density=True, label="real")
    axes[1].hist(synthetic_arrays["X_acc"].reshape(-1), bins=80, alpha=0.5, density=True, label="synthetic")
    axes[1].set_title("ACC distribution")
    axes[1].legend()

    axes[2].hist(real_arrays["X_slow"].reshape(-1), bins=80, alpha=0.5, density=True, label="real")
    axes[2].hist(synthetic_arrays["X_slow"].reshape(-1), bins=80, alpha=0.5, density=True, label="synthetic")
    axes[2].set_title("EDA/TEMP distribution")
    axes[2].legend()

    fig.tight_layout()
    finalize_plot(fig, path, save_plot)


def plot_synthetic_examples(
    X_bvp: np.ndarray,
    X_acc: np.ndarray,
    X_slow: np.ndarray,
    y: np.ndarray,
    subjects: np.ndarray,
    method_name: str,
    train_config: KoVAETrainConfig,
    path: Path,
    save_plot: bool,
    max_examples: int,
) -> None:
    n_examples = min(max_examples, len(y))
    fig, axes = plt.subplots(n_examples, 3, figsize=(15, 4 * n_examples))

    if n_examples == 1:
        axes = np.expand_dims(axes, axis=0)

    for i in range(n_examples):
        activity_label = int(y[i])
        subject = str(subjects[i])

        t_bvp = np.arange(train_config.bvp_len) / train_config.bvp_hz
        axes[i, 0].plot(t_bvp, X_bvp[i, :, 0])
        axes[i, 0].set_title(f"{method_name} | BVP | {subject} | label {activity_label}")
        axes[i, 0].set_xlabel("Time (sec)")

        t_acc = np.arange(train_config.acc_len) / train_config.acc_hz
        axes[i, 1].plot(t_acc, X_acc[i, :, 0], label="ACC_x")
        axes[i, 1].plot(t_acc, X_acc[i, :, 1], label="ACC_y")
        axes[i, 1].plot(t_acc, X_acc[i, :, 2], label="ACC_z")
        axes[i, 1].set_title("Synthetic ACC")
        axes[i, 1].set_xlabel("Time (sec)")
        axes[i, 1].legend()

        t_slow = np.arange(train_config.slow_len) / train_config.slow_hz
        axes[i, 2].plot(t_slow, X_slow[i, :, 0], label="EDA")
        axes[i, 2].plot(t_slow, X_slow[i, :, 1], label="TEMP")
        axes[i, 2].set_title("Synthetic EDA/TEMP")
        axes[i, 2].set_xlabel("Time (sec)")
        axes[i, 2].legend()

    fig.tight_layout()
    finalize_plot(fig, path, save_plot)


# ============================================================
# Save outputs for one method
# ============================================================

def save_method_outputs(
    method_name: str,
    method_paths: Dict[str, Path],
    X_bvp_syn: np.ndarray,
    X_acc_syn: np.ndarray,
    X_slow_syn: np.ndarray,
    y_syn: np.ndarray,
    subjects_syn: np.ndarray,
    metadata_df: pd.DataFrame,
    summary: dict,
    stability_summary: dict,
    real_arrays: Dict[str, np.ndarray],
    train_config: KoVAETrainConfig,
    gen_config: GenerationConfig,
    A_raw: np.ndarray,
    A_stable: np.ndarray,
) -> None:
    np.save(method_paths["synthetic"] / "generated_subjects_X_bvp_64hz.npy", X_bvp_syn)
    np.save(method_paths["synthetic"] / "generated_subjects_X_acc_32hz.npy", X_acc_syn)
    np.save(method_paths["synthetic"] / "generated_subjects_X_slow_4hz.npy", X_slow_syn)
    np.save(method_paths["synthetic"] / "generated_subjects_all_y.npy", y_syn)
    np.save(method_paths["synthetic"] / "generated_subjects_all_subject.npy", subjects_syn)

    np.savez_compressed(
        method_paths["synthetic"] / "generated_subjects_native_rate_arrays.npz",
        X_bvp_64hz=X_bvp_syn,
        X_acc_32hz=X_acc_syn,
        X_slow_4hz=X_slow_syn,
        y=y_syn,
        subject=subjects_syn,
    )

    metadata_df.to_csv(method_paths["synthetic"] / "generated_subjects_metadata.csv", index=False)

    save_json(summary, method_paths["results"] / "generation_summary.json")
    save_json(stability_summary, method_paths["results"] / "koopman_stability.json")
    np.save(method_paths["results"] / "koopman_matrix_raw.npy", A_raw)
    np.save(method_paths["results"] / "koopman_matrix_stable.npy", A_stable)

    pd.DataFrame(
        [{"activity_label": k, "count": v} for k, v in sorted(summary["activity_counts"].items())]
    ).to_csv(method_paths["results"] / "synthetic_activity_distribution.csv", index=False)

    pd.DataFrame(
        [{"synthetic_subject": k, "count": v} for k, v in sorted(summary["subject_counts"].items())]
    ).to_csv(method_paths["results"] / "synthetic_subject_counts.csv", index=False)

    synthetic_arrays = {
        "X_bvp": X_bvp_syn,
        "X_acc": X_acc_syn,
        "X_slow": X_slow_syn,
    }

    plot_koopman_eigenvalues_comparison(
        A_raw=A_raw,
        A_stable=A_stable,
        path=method_paths["figures"] / "raw_vs_stable_koopman_eigenvalues.png",
        save_plot=gen_config.save_plot,
    )

    plot_synthetic_examples(
        X_bvp=X_bvp_syn,
        X_acc=X_acc_syn,
        X_slow=X_slow_syn,
        y=y_syn,
        subjects=subjects_syn,
        method_name=method_name,
        train_config=train_config,
        path=method_paths["figures"] / "synthetic_subject_examples.png",
        save_plot=gen_config.save_plot,
        max_examples=gen_config.max_plot_examples,
    )

    plot_activity_distribution(
        y=y_syn,
        path=method_paths["figures"] / "synthetic_activity_distribution.png",
        save_plot=gen_config.save_plot,
    )

    plot_branch_histograms(
        real_arrays=real_arrays,
        synthetic_arrays=synthetic_arrays,
        path=method_paths["figures"] / "real_vs_synthetic_value_histograms.png",
        save_plot=gen_config.save_plot,
    )


# ============================================================
# Run one generation method
# ============================================================

def run_single_generation_method(
    method_name: str,
    base_paths: Dict[str, Path],
    model: MultiBranchKoVAE,
    arrays: Dict[str, np.ndarray],
    train_config: KoVAETrainConfig,
    gen_config: GenerationConfig,
    activity_to_idx: Dict[str, int],
    subject_to_idx: Dict[str, int],
    y_train_encoded: np.ndarray,
    latent_banks: Dict[str, Dict[int, np.ndarray]],
    A_raw: np.ndarray,
    A_stable: np.ndarray,
    stability_summary: dict,
    device: torch.device,
    logger: logging.Logger,
    rng: np.random.Generator,
) -> Dict[str, object]:
    method_paths = get_method_paths(base_paths, method_name)
    create_dirs(method_paths)

    logger.info("=" * 80)
    logger.info("Running generation method: %s", method_name)
    logger.info("Synthetic folder: %s", method_paths["synthetic"])
    logger.info("Results folder: %s", method_paths["results"])
    logger.info("Figures folder: %s", method_paths["figures"])

    X_bvp_all = []
    X_acc_all = []
    X_slow_all = []
    y_all = []
    subjects_all = []
    metadata_rows = []
    synthetic_global_id = 0

    idx_to_activity_label = {idx: int(label) for label, idx in activity_to_idx.items()}

    for subject_num in range(1, gen_config.num_synthetic_subjects + 1):
        synthetic_subject_id = f"{gen_config.synthetic_subject_prefix}_{method_name}_{subject_num:03d}"

        subject_style, style_metadata = sample_synthetic_subject_style(
            model=model,
            subject_to_idx=subject_to_idx,
            rng=rng,
            generation_config=gen_config,
            device=device,
        )

        subject_activity_encoded = sample_activity_sequence(
            rng=rng,
            y_train_encoded=y_train_encoded,
            n=gen_config.windows_per_subject,
            strategy=gen_config.activity_sampling_strategy,
        )

        logger.info("Generating subject %s with %d windows", synthetic_subject_id, gen_config.windows_per_subject)

        for start in range(0, gen_config.windows_per_subject, gen_config.generation_batch_size):
            end = min(start + gen_config.generation_batch_size, gen_config.windows_per_subject)
            activity_batch = subject_activity_encoded[start:end]

            if method_name == "rollout_v1":
                z = generate_latent_rollout_batch(
                    activity_batch=activity_batch,
                    z0_banks=latent_banks["z0_banks"],
                    A_stable=A_stable,
                    train_config=train_config,
                    generation_config=gen_config,
                    rng=rng,
                    device=device,
                )
            elif method_name == "posterior_bank_v2":
                z = generate_posterior_bank_batch(
                    activity_batch=activity_batch,
                    trajectory_banks=latent_banks["trajectory_banks"],
                    A_stable=A_stable,
                    generation_config=gen_config,
                    rng=rng,
                    device=device,
                )
            else:
                raise ValueError(f"Unknown generation method: {method_name}")

            decoded = decode_synthetic_batch(
                model=model,
                z=z,
                activity_batch=activity_batch,
                subject_style=subject_style,
                device=device,
            )

            X_bvp_all.append(decoded["bvp"])
            X_acc_all.append(decoded["acc"])
            X_slow_all.append(decoded["slow"])

            original_labels = np.asarray([idx_to_activity_label[int(a)] for a in activity_batch], dtype=np.int64)
            y_all.append(original_labels)
            subjects_all.append(np.asarray([synthetic_subject_id] * len(activity_batch), dtype=object))

            for local_i, encoded_activity in enumerate(activity_batch):
                metadata_rows.append({
                    "synthetic_global_id": int(synthetic_global_id),
                    "synthetic_subject": synthetic_subject_id,
                    "window_in_subject": int(start + local_i),
                    "activity_encoded": int(encoded_activity),
                    "activity_label": int(idx_to_activity_label[int(encoded_activity)]),
                    "generation_method": method_name,
                    "generation_mode": "stable_koopman_rollout" if method_name == "rollout_v1" else "posterior_bank_with_koopman_guidance",
                    "raw_spectral_radius": stability_summary["raw_spectral_radius"],
                    "stable_spectral_radius": stability_summary["stable_spectral_radius"],
                    "koopman_scaling_factor": stability_summary["scaling_factor"],
                    **style_metadata,
                })
                synthetic_global_id += 1

    X_bvp_syn = np.concatenate(X_bvp_all, axis=0).astype(np.float32)
    X_acc_syn = np.concatenate(X_acc_all, axis=0).astype(np.float32)
    X_slow_syn = np.concatenate(X_slow_all, axis=0).astype(np.float32)
    y_syn = np.concatenate(y_all, axis=0).astype(np.int64)
    subjects_syn = np.concatenate(subjects_all, axis=0).astype(str)

    metadata_df = pd.DataFrame(metadata_rows)

    activity_counts = {int(k): int(v) for k, v in zip(*np.unique(y_syn, return_counts=True))}
    subject_counts = {str(k): int(v) for k, v in zip(*np.unique(subjects_syn, return_counts=True))}

    summary = {
        "generation_method": method_name,
        "num_generated_windows": int(len(y_syn)),
        "num_synthetic_subjects": int(gen_config.num_synthetic_subjects),
        "windows_per_subject": int(gen_config.windows_per_subject),
        "X_bvp_shape": list(X_bvp_syn.shape),
        "X_acc_shape": list(X_acc_syn.shape),
        "X_slow_shape": list(X_slow_syn.shape),
        "y_shape": list(y_syn.shape),
        "subject_shape": list(subjects_syn.shape),
        "activity_counts": activity_counts,
        "subject_counts": subject_counts,
        "synthetic_output_folder": str(method_paths["synthetic"]),
        "results_folder": str(method_paths["results"]),
        "figures_folder": str(method_paths["figures"]),
        "outputs_are_normalized": True,
        "save_plot": bool(gen_config.save_plot),
    }

    save_method_outputs(
        method_name=method_name,
        method_paths=method_paths,
        X_bvp_syn=X_bvp_syn,
        X_acc_syn=X_acc_syn,
        X_slow_syn=X_slow_syn,
        y_syn=y_syn,
        subjects_syn=subjects_syn,
        metadata_df=metadata_df,
        summary=summary,
        stability_summary=stability_summary,
        real_arrays=arrays,
        train_config=train_config,
        gen_config=gen_config,
        A_raw=A_raw,
        A_stable=A_stable,
    )

    logger.info("Finished method %s | generated windows: %d", method_name, len(y_syn))

    return summary


# ============================================================
# Main dual-method pipeline
# ============================================================

def run_dual_generation(gen_config: GenerationConfig) -> Dict[str, object]:
    base_paths = get_base_paths(gen_config)
    create_dirs(base_paths)
    set_random_seed(gen_config.random_seed)

    logger = setup_logging(base_paths["logs"] / "kovae_generation_dual_methods.log")
    save_json(asdict(gen_config), base_paths["configs"] / "kovae_generation_dual_methods_config.json")

    device = get_device()
    rng = np.random.default_rng(gen_config.random_seed)

    logger.info("Starting dual-method KoVAE generation")
    logger.info("Methods to run: %s", gen_config.methods_to_run)
    logger.info("Device: %s", device)
    logger.info("Save plots: %s", gen_config.save_plot)

    if not base_paths["checkpoint"].exists():
        raise FileNotFoundError(f"Checkpoint not found: {base_paths['checkpoint']}")
    if not base_paths["koopman_matrix"].exists():
        raise FileNotFoundError(f"Koopman matrix not found: {base_paths['koopman_matrix']}")

    checkpoint = load_checkpoint(base_paths["checkpoint"], device=device)
    train_config = config_from_checkpoint(checkpoint["config"])

    activity_to_idx = {str(k): int(v) for k, v in checkpoint["activity_to_idx"].items()}
    subject_to_idx = {str(k): int(v) for k, v in checkpoint["subject_to_idx"].items()}

    model = MultiBranchKoVAE(
        config=train_config,
        num_activities=checkpoint["num_activities"],
        num_subject_tokens=checkpoint["num_subject_tokens"],
    ).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    arrays = load_native_rate_arrays(base_paths["processed"])
    y_encoded = encode_activities(arrays["y"], activity_to_idx)
    train_indices = build_indices_by_subject(arrays["subjects"], train_config.train_subjects)
    y_train_encoded = y_encoded[train_indices]

    A_raw = np.load(base_paths["koopman_matrix"]).astype(np.float32)
    A_stable, stability_summary = stabilize_koopman_matrix(
        A=A_raw,
        target_radius=gen_config.target_spectral_radius,
        stabilize=gen_config.stabilize_koopman,
    )

    latent_banks = collect_activity_latent_banks(
        model=model,
        arrays=arrays,
        train_config=train_config,
        activity_to_idx=activity_to_idx,
        subject_to_idx=subject_to_idx,
        device=device,
        logger=logger,
    )

    # Save shared banks once at the base results folder
    np.savez_compressed(
        base_paths["results_base"] / "shared_activity_z0_banks.npz",
        **{f"activity_{k}_z0_bank": v for k, v in latent_banks["z0_banks"].items()}
    )
    np.savez_compressed(
        base_paths["results_base"] / "shared_activity_trajectory_banks.npz",
        **{f"activity_{k}_trajectory_bank": v for k, v in latent_banks["trajectory_banks"].items()}
    )

    method_summaries = {}

    for method_name in gen_config.methods_to_run:
        summary = run_single_generation_method(
            method_name=method_name,
            base_paths=base_paths,
            model=model,
            arrays=arrays,
            train_config=train_config,
            gen_config=gen_config,
            activity_to_idx=activity_to_idx,
            subject_to_idx=subject_to_idx,
            y_train_encoded=y_train_encoded,
            latent_banks=latent_banks,
            A_raw=A_raw,
            A_stable=A_stable,
            stability_summary=stability_summary,
            device=device,
            logger=logger,
            rng=rng,
        )
        method_summaries[method_name] = summary

    combined_summary = {
        "methods_run": gen_config.methods_to_run,
        "save_plot": bool(gen_config.save_plot),
        "shared_base_folders": {
            "synthetic_base": str(base_paths["synthetic_base"]),
            "results_base": str(base_paths["results_base"]),
            "figures_base": str(base_paths["figures_base"]),
        },
        "shared_koopman_stability": stability_summary,
        "method_summaries": method_summaries,
    }

    save_json(combined_summary, base_paths["results_base"] / "combined_generation_summary.json")

    logger.info("Dual-method generation completed.")
    return combined_summary


# ============================================================
# Script entry point
# ============================================================

if __name__ == "__main__":
    outputs = run_dual_generation(GEN_CONFIG)
    print(json.dumps(outputs, indent=2))


2026-07-07 00:17:31 | INFO | Starting dual-method KoVAE generation
2026-07-07 00:17:31 | INFO | Methods to run: ['rollout_v1', 'posterior_bank_v2']
2026-07-07 00:17:31 | INFO | Device: cuda
2026-07-07 00:17:31 | INFO | Save plots: True
2026-07-07 00:17:33 | INFO | Activity 0 | z0 bank (3032, 32) | trajectory bank (3032, 64, 32)
2026-07-07 00:17:33 | INFO | Activity 1 | z0 bank (2139, 32) | trajectory bank (2139, 64, 32)
2026-07-07 00:17:33 | INFO | Activity 2 | z0 bank (1500, 32) | trajectory bank (1500, 64, 32)
2026-07-07 00:17:33 | INFO | Activity 3 | z0 bank (2296, 32) | trajectory bank (2296, 64, 32)
2026-07-07 00:17:33 | INFO | Activity 4 | z0 bank (4580, 32) | trajectory bank (4580, 64, 32)
2026-07-07 00:17:33 | INFO | Activity 5 | z0 bank (8792, 32) | trajectory bank (8792, 64, 32)
2026-07-07 00:17:33 | INFO | Activity 6 | z0 bank (2903, 32) | trajectory bank (2903, 64, 32)
2026-07-07 00:17:33 | INFO | Activity 7 | z0 bank (5520, 32) | trajectory bank (5520, 64, 32)
2026-07-07 0

{
  "methods_run": [
    "rollout_v1",
    "posterior_bank_v2"
  ],
  "save_plot": true,
  "shared_base_folders": {
    "synthetic_base": "/home/iailab42/khans1/projects/ir/data/synthetic_subjects/kovae",
    "results_base": "/home/iailab42/khans1/projects/ir/results/kovae_generation",
    "figures_base": "/home/iailab42/khans1/projects/ir/figures/kovae_generation"
  },
  "shared_koopman_stability": {
    "raw_spectral_radius": 1.4776456356048584,
    "target_spectral_radius": 0.98,
    "stable_spectral_radius": 0.9800000190734863,
    "stabilize_koopman": true,
    "scaling_factor": 0.6632171992974808,
    "stability_action": "scaled_matrix"
  },
  "method_summaries": {
    "rollout_v1": {
      "generation_method": "rollout_v1",
      "num_generated_windows": 5000,
      "num_synthetic_subjects": 10,
      "windows_per_subject": 500,
      "X_bvp_shape": [
        5000,
        512,
        1
      ],
      "X_acc_shape": [
        5000,
        256,
        3
      ],
      "X_slow_

## Run both generation methods

Can keep both methods enabled:

```python
GEN_CONFIG.methods_to_run = ["rollout_v1", "posterior_bank_v2"]
```

Or run only one:

```python
GEN_CONFIG.methods_to_run = ["rollout_v1"]
GEN_CONFIG.methods_to_run = ["posterior_bank_v2"]
```

Plot switch:

```python
GEN_CONFIG.save_plot = True   # save plots
GEN_CONFIG.save_plot = False  # only show plots
```


In [2]:
GEN_CONFIG.methods_to_run = ["rollout_v1", "posterior_bank_v2"]
GEN_CONFIG.save_plot = True

outputs = run_dual_generation(GEN_CONFIG)
outputs


2026-07-07 00:18:57 | INFO | Starting dual-method KoVAE generation
2026-07-07 00:18:57 | INFO | Methods to run: ['rollout_v1', 'posterior_bank_v2']
2026-07-07 00:18:57 | INFO | Device: cuda
2026-07-07 00:18:57 | INFO | Save plots: True
2026-07-07 00:18:59 | INFO | Activity 0 | z0 bank (3032, 32) | trajectory bank (3032, 64, 32)
2026-07-07 00:18:59 | INFO | Activity 1 | z0 bank (2139, 32) | trajectory bank (2139, 64, 32)
2026-07-07 00:18:59 | INFO | Activity 2 | z0 bank (1500, 32) | trajectory bank (1500, 64, 32)
2026-07-07 00:18:59 | INFO | Activity 3 | z0 bank (2296, 32) | trajectory bank (2296, 64, 32)
2026-07-07 00:18:59 | INFO | Activity 4 | z0 bank (4580, 32) | trajectory bank (4580, 64, 32)
2026-07-07 00:18:59 | INFO | Activity 5 | z0 bank (8792, 32) | trajectory bank (8792, 64, 32)
2026-07-07 00:18:59 | INFO | Activity 6 | z0 bank (2903, 32) | trajectory bank (2903, 64, 32)
2026-07-07 00:18:59 | INFO | Activity 7 | z0 bank (5520, 32) | trajectory bank (5520, 64, 32)
2026-07-07 0

{'methods_run': ['rollout_v1', 'posterior_bank_v2'],
 'save_plot': True,
 'shared_base_folders': {'synthetic_base': '/home/iailab42/khans1/projects/ir/data/synthetic_subjects/kovae',
  'results_base': '/home/iailab42/khans1/projects/ir/results/kovae_generation',
  'figures_base': '/home/iailab42/khans1/projects/ir/figures/kovae_generation'},
 'shared_koopman_stability': {'raw_spectral_radius': 1.4776456356048584,
  'target_spectral_radius': 0.98,
  'stable_spectral_radius': 0.9800000190734863,
  'stabilize_koopman': True,
  'scaling_factor': 0.6632171992974808,
  'stability_action': 'scaled_matrix'},
 'method_summaries': {'rollout_v1': {'generation_method': 'rollout_v1',
   'num_generated_windows': 5000,
   'num_synthetic_subjects': 10,
   'windows_per_subject': 500,
   'X_bvp_shape': [5000, 512, 1],
   'X_acc_shape': [5000, 256, 3],
   'X_slow_shape': [5000, 32, 2],
   'y_shape': [5000],
   'subject_shape': [5000],
   'activity_counts': {1: 490,
    2: 334,
    3: 232,
    4: 377,
   